# 原账户回撤对 SOXL 局部净价值的固定方向排序

只读 Qlib2018–2022原架构训练账本，预定分数为负的09:46账户回撤。逐年、季度与受限状态检验必要条件，无拟合、阈值或资金账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
root=Path.cwd();card=root/'research/stage1_soxl_local_value_drawdown_rank_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
a=root/s['SOXL_card'];b=root/s['reference_card']
assert sha(root/s['overlap_plan'])==s['overlap_plan_sha256']
assert sha(a/'summary.json')==s['SOXL_summary_sha256'] and sha(b/'summary.json')==s['reference_summary_sha256']
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sa['nas_readback']['verified'] and sa['nas_readback']['sha256']==s['SOXL_NAS_sha256']
assert sb['nas_readback']['verified'] and sb['recorder_id']==s['reference_recorder_id']
assert sha(a/'qlib_recorder'/s['SOXL_experiment_id']/s['SOXL_recorder_id']/'artifacts/analysis/labels')==s['SOXL_labels_sha256']
assert sha(b/'qlib_recorder'/s['reference_experiment_id']/s['reference_recorder_id']/'artifacts/paths/reference_0')==s['reference_path_sha256']
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['SOXL_experiment_id'],recorder_id=s['SOXL_recorder_id']);labels=rec.load_object('analysis/labels')
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['reference_experiment_id'],recorder_id=s['reference_recorder_id']);bundle=rec.load_object('paths/reference_0')
path=bundle['replay'];assert len(labels)==len(path)==1061
cal=pd.DatetimeIndex(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day'))
ld=pd.DatetimeIndex(labels.trade_date);assert ld.equals(pd.DatetimeIndex(path.trade_date)) and ld.equals(cal)
assert pd.DatetimeIndex(labels.signal_date).equals(pd.DatetimeIndex(path.signal_date))
assert labels.soxl_local_fee_net_advantage.notna().sum()==688
x=pd.DataFrame({'trade_date':ld,'year':ld.year,'quarter':ld.quarter,'risk_state':path.risk_state.to_numpy(int),
                'cap_binding':labels.cap_binding.to_numpy(bool),'score':-path.portfolio_drawdown.to_numpy(float),
                'eligible':labels.eligible.to_numpy(bool),'label':labels.soxl_local_fee_net_advantage.to_numpy(float)})
assert np.isfinite(x.score).all() and x.loc[x.label.notna(),'eligible'].all()
valid=x[x.label.notna()].copy()
def rank(g):
 k=g[g.label.notna()].copy();z=k[k.label.abs()>1e-12];y=(z.label>0).to_numpy(int)
 auc=float(roc_auc_score(y,z.score.to_numpy(float))) if len(set(y))==2 else None
 sp=float(spearmanr(k.score.to_numpy(float),k.label.to_numpy(float)).statistic) if len(k)>2 and k.score.nunique()>1 and k.label.nunique()>1 else None
 return {'sessions':len(g),'mature':len(k),'positive':int((k.label>1e-12).sum()),'negative':int((k.label<-1e-12).sum()),
         'zero':int((k.label.abs()<=1e-12).sum()),'auc':auc,'spearman':sp,'mean_label':float(k.label.mean()) if len(k) else None}
full=rank(x)
annual=[{'year':int(year),'partial_endpoint':year==2018,**rank(g)} for year,g in x.groupby('year',sort=True)]
quarter=[{'year':int(year),'quarter':int(q),**rank(g)} for (year,q),g in x.groupby(['year','quarter'],sort=True)]
by_binding={str(binding).lower():rank(g) for binding,g in x.groupby('cap_binding')}
era=[{'era':'2020_2021',**rank(x[x.year.isin([2020,2021])])},{'era':'2022',**rank(x[x.year==2022])}]
binding_era=[{'era':'2020_2021',**rank(x[x.year.isin([2020,2021]) & x.cap_binding])},{'era':'2022',**rank(x[(x.year==2022)&x.cap_binding])}]
binding_state=[]
for state in range(4):
 for name,years in [('2020_2021',[2020,2021]),('2022',[2022])]:
  binding_state.append({'risk_state':state,'era':name,**rank(x[x.year.isin(years)&x.cap_binding&(x.risk_state==state)])})
assert len(annual)==5 and len(quarter)==17 and full['mature']==688 and sum(a['mature'] for a in annual)==688
required_years=[next(z for z in annual if z['year']==year) for year in (2020,2021,2022)]
gates={'each_2020_2022_auc_above_half':all(z['auc'] is not None and z['auc']>.5 for z in required_years),
       'each_2020_2022_spearman_positive':all(z['spearman'] is not None and z['spearman']>0 for z in required_years),
       'binding_era_spearman_positive':all(z['spearman'] is not None and z['spearman']>0 for z in binding_era)}
result={'status':'read_only_SOXL_local_value_drawdown_rank_necessary_condition','plan_sha256':sha(plan_path),
        'source_recorder_ids':{'labels':s['SOXL_recorder_id'],'reference':s['reference_recorder_id']},
        'full_sessions':len(x),'mature_labels':len(valid),'full':full,'annual':annual,'quarter':quarter,'by_binding':by_binding,'era':era,'binding_era':binding_era,'binding_state':binding_state,
        'gates':gates,'necessary_gate_pass':all(gates.values()),'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Historical same-next-target local value is not an own-state full f696 account','Direction emerged from reused interval attribution, so 2022 is not independent validation','No drawdown threshold or sign reversal may be chosen after this fixed screen']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'full':full,'years':[(z['year'],z['auc'],z['spearman']) for z in annual],'binding_era':binding_era,'gates':gates})


{'full': {'sessions': 1061, 'mature': 688, 'positive': 345, 'negative': 311, 'zero': 32, 'auc': 0.5281886387995713, 'spearman': 0.005420155518317013, 'mean_label': 4.957468748976795e-05}, 'years': [(2018, 0.6704545454545454, -0.021052631578947368), (2019, 0.5359977324263039, 0.053347953182907595), (2020, 0.43097814988768635, -0.252355498030503), (2021, 0.49753694581280783, -0.07365077045500472), (2022, 0.4487153820487153, -0.09730296205926665)], 'binding_era': [{'era': '2020_2021', 'sessions': 238, 'mature': 238, 'positive': 139, 'negative': 99, 'zero': 0, 'auc': 0.44953128406365805, 'spearman': -0.13691101142538475, 'mean_label': 0.0002783639726661111}, {'era': '2022', 'sessions': 246, 'mature': 245, 'positive': 111, 'negative': 133, 'zero': 1, 'auc': 0.4551920341394025, 'spearman': -0.09922643187622911, 'mean_label': -0.00016301961900291737}], 'gates': {'each_2020_2022_auc_above_half': False, 'each_2020_2022_spearman_positive': False, 'binding_era_spearman_positive': False}}


## 中文结论

# 连续回撤因子在逐年局部价值排序中失败

只读封存Qlib原架构2018–2022训练账户的1061个交易日、688个成熟现金转SOXL含Tiered费局部标签。固定方向分数为**负的原账户09:46回撤幅度**：回撤越深，预测次日现金部署价值越低。核对来源SHA、NAS收据/限定回读、Qlib日历与交易日；保留部分2018、完整2019–2022、17个季度、受限/未受限层及风险状态0–3。无拟合、阈值或新资金账户。

全段正/负/零局部标签分别 **345/311/32**，按非零标签计算的AUC **0.5282**，连续局部价值Spearman仅 **+0.0054**。关键的完整2020/2021/2022年AUC分别 **0.4310/0.4975/0.4487**，Spearman **−0.2524/−0.0737/−0.0973**；全部与预定正向必要条件冲突。受限层2020/2021合并期与2022期Spearman **−0.1369/−0.0992**，也均不为正。年度、季度及每个原风险状态细项见 `result.json`。

跨年合并的微弱正AUC不能掩盖逐年反向排序。上一卡发现2022在同风险状态下回撤较深、局部价值较差；本卡说明这种**年份间均值差不能转成同一年逐日部署因子**。固定版本 `not_promoted_fixed_negative_drawdown_local_value_rank_necessary_gate_failed`，不翻转方向、不用2022年阈值拟合、不进入信息模型或精确f696完整账户。旧风险状态和连续回撤都无法单独解释2020/2021机会与2022保护的切换；需要实质不同的事前信息，并先验证其跨年、分层的局部净价值增量。所有历史标签仍非独立前向，SOXX v2目标未达成。
